# PyTorch Refresher


Follows Sebastian Rashcka's excellent [blog post found here](https://sebastianraschka.com/teaching/pytorch-1h).

## Understaning tensors

### Scalars, vectors matrices and tensors

In [1]:
import torch

# 0D tensor (scalar)
tensor0d = torch.tensor(1)

# 1d tensor (vector)
tensor1d = torch.tensor([1, 2, 3])

# 2d tensor (matrix)
tensor2d = torch.tensor([[1, 2], [3, 4]])

# 3d tensor (tensor)
tensor = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]]) 

print(f"Scalar: {tensor0d}")
print(f"Vector: {tensor1d}")
print(f"Matrix: {tensor2d}")
print(f"Tensor: {tensor}")

Scalar: 1
Vector: tensor([1, 2, 3])
Matrix: tensor([[1, 2],
        [3, 4]])
Tensor: tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])


### Tensor data types

In [2]:
print(f"Data type: {tensor1d.dtype}")
print(f"Data type: {torch.tensor([1.0, 2.0, 3.0]).dtype}")
print(f"Data type: {tensor1d.to(torch.float32).dtype}")

Data type: torch.int64
Data type: torch.float32
Data type: torch.float32


### Common PyTorch tensor operations

In [3]:
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])

print(f"Tensor shape: {tensor2d.shape}")

Tensor shape: torch.Size([2, 3])


`[2, 3]` means the tensor has 2 rows and 3 columns.

In [4]:
print(f"Tensor reshaped: \n{tensor2d.reshape(3, 2)}")

Tensor reshaped: 
tensor([[1, 2],
        [3, 4],
        [5, 6]])


It is more common to use `.view`. `.reshape` will copy if it cannot use the same memory.

In [5]:
print(f"Tensor reshaped: \n{tensor2d.view(3, 2)}")

Tensor reshaped: 
tensor([[1, 2],
        [3, 4],
        [5, 6]])


The transpose can be taken with `.T`. This is a view.

In [6]:
print(f"Tensor reshaped: \n{tensor2d.T}")

Tensor reshaped: 
tensor([[1, 4],
        [2, 5],
        [3, 6]])


There are two ways of multiplying two matrices.

1. `.matmul`
2. `@` operator

In [7]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [8]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

## Seeing models as computation graphs

A computational graph is a direction graph that allows the expression and visualisation of mathematical expressions.
In the context of DL, a computational graph laysout the sequence of calculations neeed to cojmpute the output of a neural network.

![nerual net computational graph](https://sebastianraschka.com/images/teaching/pytorch-1h/figure_07.webp)

In [9]:
import torch.nn.functional as F

y = torch.tensor([1.0]) # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)

print(f"Input: {x1}")
print(f"Output: {a}")
print(f"Actual: {y}")
print(f"Loss: {loss:.4f}")

Input: tensor([1.1000])
Output: tensor([0.9183])
Actual: tensor([1.])
Loss: 0.0852


## Automatic differentiation made easy

![auto grad on computational graph](https://sebastianraschka.com/images/teaching/pytorch-1h/figure_08.webp)

Gradients are needed when doing backpropagation when training neural networks.

By tracking every operation performed on tensors, PyTorch's autograd engine contructs a computation graph in the background.
When calling the grad function, the gradient of the loss with respect to the model parameter `w1` can be computed.

In [10]:
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
# NOTE: requires_grad has been added
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)


grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(f"Gradient with respect to w1: {grad_L_w1}")
print(f"Gradient with respect to b: {grad_L_b}")

Gradient with respect to w1: (tensor([-0.0898]),)
Gradient with respect to b: (tensor([-0.0817]),)


The above has been done manually, which can be useful for experimentation. In practice, PyTorch provides even more high-level tools to automate this process.
For example, `.backward` can be called on the loss. PyTorch will compute the gradients of all the leaf nodes in the graph, which will be stored via the tensors' `.grad` attributes.

In [11]:
loss.backward()

print(f"Gradient with respect to w1: {w1.grad}")
print(f"Gradient with respect to b: {b.grad}")

Gradient with respect to w1: tensor([-0.0898])
Gradient with respect to b: tensor([-0.0817])


## Implementing mulilayer neural networks

When implementing neural networks in PyTorch, typically `torch.nn.Module` is subclassed.
This `Module` base class provides a lot of functionality. For example, it allows the encapsulation of layers and operations and keep track of the model's performance.

Within the subclass, the network layers are defined in the `__init__` constructor and specify how they interact in the `forward` method.
The `forward` method describes how the input data passes through the network and comes together as a computational graph.

The backwards mehtod is not typically needed to be implemented.

In [12]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_ouputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_ouputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [13]:
model = NeuralNetwork(num_inputs=50, num_ouputs=3)
print(f"Model: {model}")

Model: NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Note that using `Sequential` means that only `self.layers` needs to be called instead of calling each layer individualy.

In [14]:
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Total num of trainable model parameters: {num_params}")

Total num of trainable model parameters: 2213


Note that each param for which `requires_grad=True` counts as a trainable parameter and will be updated during training.

A *linear layer* multiplies the inputs with a weight matrix and adds a bias vector. This is sometimes referred to as a *feedforward* or *full connected layer*.

The corresponding weight parmater matrix can be accessed as follows:

In [15]:
print(f"First layer weights: {model.layers[0].weight}")
print(f"First layer weights shape: {model.layers[0].weight.shape}")

First layer weights: Parameter containing:
tensor([[ 0.0985,  0.0283,  0.0067,  ..., -0.0278, -0.0038, -0.0842],
        [ 0.1045, -0.0222,  0.1115,  ...,  0.1365,  0.0628, -0.0588],
        [-0.1248, -0.0758, -0.1379,  ..., -0.0188,  0.0451,  0.0388],
        ...,
        [ 0.0994, -0.0255, -0.0205,  ..., -0.1361,  0.0285, -0.0930],
        [-0.0429,  0.1041,  0.0474,  ..., -0.0688, -0.0302, -0.0733],
        [ 0.1311,  0.0911, -0.0436,  ..., -0.1283, -0.1286,  0.0780]],
       requires_grad=True)
First layer weights shape: torch.Size([30, 50])


The bias can be accessed like this:

In [16]:
print(f"First layer biases: \n{model.layers[0].bias}")
print(f"First layer biases shape: {model.layers[0].bias.shape}")

First layer biases: 
Parameter containing:
tensor([-0.0133,  0.1369, -0.0056, -0.0745, -0.0304,  0.0694,  0.0263,  0.0465,
        -0.0667, -0.1413,  0.0633, -0.0144,  0.0909, -0.0106,  0.0698, -0.0769,
        -0.1177, -0.0257, -0.0367,  0.1149,  0.0092, -0.0647,  0.0713,  0.1040,
         0.0474, -0.0436,  0.1130, -0.0683, -0.0629,  0.0155],
       requires_grad=True)
First layer biases shape: torch.Size([30])


These numbers are different each time, as they are randomised. To control this, use `torch.manual_seed`.

In [17]:
torch.manual_seed(123)

model = NeuralNetwork(num_inputs=50, num_ouputs=3)

print(f"First layer weights: {model.layers[0].weight}")
print(f"First layer weights shape: {model.layers[0].weight.shape}")

First layer weights: Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)
First layer weights shape: torch.Size([30, 50])


To use the nerual network via the foward pass:

In [18]:
torch.manual_seed(123)

X = torch.rand((1, 50))
out = model(X)

print(f"Input: {X}")
print(f"Output logits: {out}")

Input: tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])
Output logits: tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


When calling `model(X)`, the forward pass of the model is automatically called.

`grad_fn=<AddmmBackward0>` represents the last-used function to compute a variable in the computational graph. PyTorch will use this informaiton when it computes gradients during backpropagation.
`Addmm` stands for matrix multiplication, followed by an additon (`Add`)

If using the network without training or backpropagation, for example, just for prediction after training, contricuting this computational graph for backpropagation can be wasteful. This is because it performs unnecessary computations and consumes additional memory. So it's best practice to use `torch.no_grad()`. This tells PyTorch taht it doesn't need to keep track of the gradients. This can be a significant saving in memory and computation.

In [19]:
with torch.no_grad():
    out = model(X)

print(f"Output logits: {out}")

Output logits: tensor([[-0.1262,  0.1080, -0.1792]])


In PyTorch, it's common to code models so that they return the outputs of the last layer (`logits`) without passing them to a nonlinear activation function.
That's because PyTorch's commonly used loss functions combine the softmax (or sigmoid for binary classification) operation with the negative log-likelihood loss in a single class.

The reason for this is numerical efficiency and stability. Because of this, the softmax function has to be called explicitly:

In [20]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)


print(f"Output: {out}")

Output: tensor([[0.3113, 0.3934, 0.2952]])


The values can now be interpreted as class-membership probs that sum up to 1.